# NLP Practical 6 — Discourse & Knowledge

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** lexical cohesion detection, rule-based pronoun resolution, RST-style discourse relation tagging via connectives, and a semantic network with forward-chaining inheritance reasoning.


In [ ]:
!pip install nltk -q
import nltk
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")


In [ ]:
# ============================================================
# PART A: Lexical cohesion detection (word-overlap based)
# ============================================================
import nltk
from collections import Counter

text = """The company launched a new product. The product received positive reviews.
Customers loved the product design. Sales increased sharply after the launch."""

sentences = nltk.sent_tokenize(text)
tokenized = [set(w.lower() for w in nltk.word_tokenize(s) if w.isalpha()) for s in sentences]

print("Lexical overlap between consecutive sentences (cohesion signal):")
for i in range(len(tokenized) - 1):
    overlap = tokenized[i] & tokenized[i + 1]
    print(f"S{i+1}-S{i+2} shared words: {overlap if overlap else '(none)'}")


In [ ]:
# ============================================================
# PART B: Rule-based pronoun resolution (Hobbs-style heuristic, simplified)
# ============================================================
import nltk

text = "Priya met Rohan at the cafe. She gave him a book."
sentences = nltk.sent_tokenize(text)

def find_candidates(sentence):
    tagged = nltk.pos_tag(nltk.word_tokenize(sentence))
    return [w for w, t in tagged if t == "NNP"]

def resolve_pronoun(pronoun, prior_sentence):
    candidates = find_candidates(prior_sentence)
    # Simplified heuristic: "she"/"her" -> most recent female-coded name (toy list),
    # "he"/"him" -> most recent male-coded name (toy list). A real system uses
    # gender/number agreement + salience ranking (Hobbs algorithm / centering theory).
    female_names, male_names = {"Priya"}, {"Rohan"}
    if pronoun.lower() in ("she", "her"):
        matches = [c for c in candidates if c in female_names]
    elif pronoun.lower() in ("he", "him"):
        matches = [c for c in candidates if c in male_names]
    else:
        matches = candidates
    return matches[-1] if matches else None

print("Sentence 1:", sentences[0])
print("Sentence 2:", sentences[1])
print("\n'She' resolves to:", resolve_pronoun("She", sentences[0]))
print("'him' resolves to:", resolve_pronoun("him", sentences[0]))


In [ ]:
# ============================================================
# PART C: RST-style discourse relation tagging (rule-based, via connectives)
# ============================================================
connective_map = {
    "because": "Cause", "since": "Cause",
    "but": "Contrast", "however": "Contrast",
    "and": "Sequence/Joint", "then": "Sequence",
    "although": "Concession", "so that": "Purpose",
    "for example": "Elaboration",
}

discourse_text = [
    "The market fell sharply because investors panicked.",
    "Sales were strong, but profits declined.",
    "The team trained hard, so that they could win the finals.",
]

for sent in discourse_text:
    found = [rel for conn, rel in connective_map.items() if conn in sent.lower()]
    relation = found if found else ["Elaboration (default)"]
    print(f"{sent}\n  -> Discourse relation(s): {relation}")


In [ ]:
# ============================================================
# PART D: Semantic network + forward-chaining reasoning
# ============================================================
# A minimal IS-A / property inheritance network with a forward-chaining rule engine.

facts = {
    ("dog", "is_a", "mammal"),
    ("mammal", "is_a", "animal"),
    ("mammal", "has_property", "warm_blooded"),
    ("animal", "has_property", "alive"),
}

rules = [
    # If X is_a Y and Y has_property Z, then X has_property Z  (property inheritance)
    lambda kb: {("dog", "has_property", z) for (y1, r1, z) in kb if r1 == "has_property"
                for (x, r2, y2) in kb if r2 == "is_a" and y2 == y1 for y1 in [y1]}
]

def forward_chain(kb, steps=3):
    kb = set(kb)
    for _ in range(steps):
        new_facts = set()
        for (x, rel, y) in list(kb):
            if rel == "is_a":
                for (y2, rel2, z) in kb:
                    if y2 == y and rel2 == "has_property":
                        new_facts.add((x, "has_property", z))
        if new_facts <= kb:
            break
        kb |= new_facts
    return kb

inferred = forward_chain(facts)
print("Inferred facts about 'dog':")
for f in sorted(inferred):
    if f[0] == "dog":
        print(" ", f)
